In [1]:
import numpy as np
import pandas as pd

In [2]:
df = pd.read_csv("IDSP 2024 report  - Sheet1.csv")
print(df.head())

           Unique ID        State/UT                         District  \
0  AP/KRS/2024/01/01  Andhra Pradesh  Nandamuri Taraka Rama Rao (NTR)   
1  AS/SBS/2024/01/02           Assam                        Charaideo   
2  AS/DAR/2024/01/03           Assam                          Darrang   
3  BH/JAH/2024/01/04           Bihar                        Jehanabad   
4  GJ/AHM/2024/01/05         Gujarat                        Ahmedabad   

           Disease/Illness  No. of Cases No. of Deaths  \
0  Acute Diarrheal Disease            30             0   
1            Leptospirosis             1             1   
2               Chickenpox             6             0   
3               Chickenpox             8             0   
4  Acute Diarrheal Disease            16             0   

  Date of Start of Outbreak    Date of Reporting      Current Status  \
0                30-12-2023  2024-03-01 00:00:00       Under Control   
1       2024-02-01 00:00:00  2024-02-01 00:00:00  Under Surveillance

In [3]:
print(df.isnull().sum())

Unique ID                      0
State/UT                       0
District                       0
Disease/Illness                0
No. of Cases                   0
No. of Deaths                  0
Date of Start of Outbreak      0
Date of Reporting            324
Current Status                 0
Comments/Action Taken          0
Week                           0
dtype: int64


In [4]:
df.fillna("", inplace = True)
print(df.isnull().sum())

Unique ID                    0
State/UT                     0
District                     0
Disease/Illness              0
No. of Cases                 0
No. of Deaths                0
Date of Start of Outbreak    0
Date of Reporting            0
Current Status               0
Comments/Action Taken        0
Week                         0
dtype: int64


In [5]:
df['State/UT'] = df['State/UT'].astype(str)
df['State/UT'] = df['State/UT'].str.strip()
df['State/UT'] = df['State/UT'].str.replace('\n', ' ', regex=True)
df['State/UT'] = df['State/UT'].str.title()

In [6]:
df['State/UT'] = df['State/UT'].replace({
    'Andhra Pradesh ': 'Andhra Pradesh',
    'Andhra Pradesh': 'Andhra Pradesh',

    'D&N Haveli And Daman And Diu': 'Dadra and Nagar Haveli and Daman and Diu',
    'Dadra And Nagar Haveli': 'Dadra and Nagar Haveli and Daman and Diu',
    'Dadra And Nagar Haveli And Daman And Diu': 'Dadra and Nagar Haveli and Daman and Diu'
})

In [7]:
print(df['State/UT'].nunique())
print(sorted(df['State/UT'].unique()))

33
['Andaman & Nicobar Islands', 'Andhra Pradesh', 'Arunachal Pradesh', 'Assam', 'Bihar', 'Chhattisgarh', 'Dadra and Nagar Haveli and Daman and Diu', 'Goa', 'Gujarat', 'Haryana', 'Himachal Pradesh', 'Jammu And Kashmir', 'Jharkhand', 'Karnataka', 'Kerala', 'Ladakh', 'Madhya Pradesh', 'Maharashtra', 'Manipur', 'Meghalaya', 'Mizoram', 'Nagaland', 'Odisha', 'Puducherry', 'Punjab', 'Rajasthan', 'Sikkim', 'Tamil Nadu', 'Telangana', 'Tripura', 'Uttar Pradesh', 'Uttarakhand', 'West Bengal']


In [8]:
all_states = sorted(df['State/UT'].unique())

In [9]:
df['Week'] = (
    df['Week']
    .astype(str)                 # convert everything to string
    .str.extract(r'(\d+)')[0]    # extract only the number
    .astype(int)                 # now convert safely
)
weekly_state = (
    df.groupby(['Week', 'State/UT'])['No. of Cases']
    .sum()
    .reset_index()
)

In [10]:
selected_week = df['Week'].max()

week_data = weekly_state[weekly_state['Week'] == selected_week].copy()

week_data.rename(columns={'No. of Cases': 'cases'}, inplace=True)

week_data = pd.DataFrame({'State/UT': all_states}).merge(
    week_data,
    on='State/UT',
    how='left'
)

week_data['cases'] = week_data['cases'].fillna(0)

week_data['Week'] = selected_week

#print(week_data)

In [11]:
# Trend Indicator
prev_week = selected_week - 1
prev_data = weekly_state[weekly_state['Week'] == prev_week][['State/UT', 'No. of Cases']]
prev_data.rename(columns={'No. of Cases': 'prev_cases'}, inplace=True)
week_data = week_data.drop(columns=['prev_cases'], errors='ignore')
week_data = week_data.merge(prev_data, on='State/UT', how='left')
week_data['prev_cases'] = week_data['prev_cases'].fillna(0)

# growth calculation
week_data['growth_pct'] = week_data.apply(
    lambda r: ((r['cases'] - r['prev_cases']) / r['prev_cases'] * 100)
    if r['prev_cases'] > 0 else 0,
    axis=1
)

In [12]:
def risk_level(row):
    if row['cases'] == 0:
        return 'No Cases'
    elif row['cases'] > 150 or row['growth_pct'] > 50:
        return 'High'
    elif row['cases'] > 50 or row['growth_pct'] > 10:
        return 'Medium'
    else:
        return 'Low'

week_data['risk_level'] = week_data.apply(risk_level, axis=1)

In [13]:
top_disease = (
    df[df['Week'] == selected_week]
    .groupby(['State/UT', 'Disease/Illness'])['No. of Cases']
    .sum()
    .reset_index()
    .sort_values(['State/UT', 'No. of Cases'], ascending=[True, False])
    .drop_duplicates('State/UT')
    .rename(columns={'Disease/Illness': 'top_disease'})
)

week_data = week_data.merge(top_disease[['State/UT', 'top_disease']], on='State/UT', how='left')

In [14]:
week_data.head()

,State/UT,Week,cases,prev_cases,growth_pct,risk_level,top_disease
0,Andaman & Nicobar Islands,52,0.0,0.0,0.000000,No Cases,NaN
1,Andhra Pradesh,52,84.0,0.0,0.000000,Medium,Acute Diarrheal Disease
2,Arunachal Pradesh,52,0.0,0.0,0.000000,No Cases,NaN
3,Assam,52,7.0,57.0,-87.719298,Low,Typhoid
4,Bihar,52,0.0,14.0,-100.000000,No Cases,NaN
